In [1]:
import numpy as np
import matplotlib.pyplot as plt


### Topic 25: Determinants (calculating high-dimensional scaling factors).

In Machine Learning, determinants are rarely used for solving linear systems. Instead, they act as the primary measure of **Probability Volume** and a diagnostic for **Dimensional Collapse**.

* **Multivariate Gaussians:** High-dimensional data (like image datasets) is often modeled using a Multivariate Gaussian distribution. The denominator scales with the determinant of the Covariance Matrix, $|\Sigma|$.
    * *The Threat:* If $|\Sigma| \to 0$, the dataset has zero variance in at least one orthogonal direction. The data volume collapses into a lower-dimensional hyperplane.
    * *The Result:* The probability density mathematically explodes to $\infty$, instantly triggering `NaN` errors during backpropagation.
* **Normalizing Flows (Generative AI):** Advanced models warp simple distributions into complex ones (e.g., generating faces). To strictly conserve probability mass across these transformations, the network calculates the determinant of the **Jacobian matrix** at every single layer.
* **The Engineering Trap (Log-Determinants):** Computing the standard determinant of a $1000 \times 1000$ matrix requires multiplying 1,000 numbers. This mathematically guarantees a `float64` underflow to $0$ or overflow to $\infty$. Deep Learning completely abandons the standard determinant, exclusively computing the **Log-Determinant** ($\log|\Sigma|$) to guarantee numerical stability.

In [2]:
A =np.random.randn(5,5)
cov_matrix = A @ A.T

# 1.The native way
standerd_det = np.linalg.det(cov_matrix)
print(f"Standard Determinant: {standerd_det:.4f}")

# 2. ml way
# slogdet returns a tuple: (sign_of_determinant, natural_log_of_determinant)
sign,log_det = np.linalg.slogdet(cov_matrix)

print(f"Sign: {sign}, Log-Determinant: {log_det:.4f}")

recoverd_det = sign * np.exp(log_det)
print(standerd_det)
print(recoverd_det)


Standard Determinant: 33.5027
Sign: 1.0, Log-Determinant: 3.5116
33.50267748973013
33.50267748973013


In [3]:
# Generate 10 random 50x50 covariance matrices
np.random.seed(42)
matrices = np.random.randn(10, 50, 50)
cov_m = matrices @ np.transpose(matrices, axes=(0, 2, 1)) # Batched A @ A.T

sign_1, log_det_1= np.linalg.slogdet(cov_m)
print(log_det_1)

smallest = np.argmin(log_det_1)
print(smallest)

[146.41473907 149.38371924 146.89657459 148.12946363 143.10739773
 143.54405334 145.70363762 144.05681335 140.39046654 139.58250634]
9


### Topic 26: Matrix Inversion Limits and the Singular Matrix Trap

The Normal Equation: Bottlenecks & Solutions

**The Exact Solution:** $$W = (X^T X)^{-1} X^T Y$$

**The 2 Engineering Bottlenecks:**
1. **$O(N^3)$ Complexity:** Matrix inversion scales cubically. Inverting a matrix with 10,000 features requires ~1 trillion operations, crushing GPU memory.
2. **The Singular Matrix Trap:** Highly correlated data (e.g., "Sq Ft" and "Sq Meters") causes perfect multi-collinearity. The determinant $|X^T X|$ drops to $0$, making it singular and crashing the script with a `LinAlgError`.

**The Machine Learning Solutions:**
* **The Pseudo-Inverse (`np.linalg.pinv`):** Uses Singular Value Decomposition (SVD) to safely approximate the inverse for *any* matrix (singular or non-square), preventing crashes.
* **Direct Solvers (`np.linalg.solve`):** When solving $Ax = b$, direct solvers compute the optimal weights algebraically without ever holding the massive $A^{-1}$ matrix in memory.

In [4]:
A_singular = np.array([
    [1,2],
    [2,4]
])

# TRAP
try:
    inverse = np.linalg.inv(A_singular)
    print(inverse)
except np.linalg.LinAlgError as e:
    print(f"Mathematic crash: {e}")

# ML (psedu-inverse)
psedu_inverse = np.linalg.pinv(A_singular)
print("psedu-inverse : ",psedu_inverse)

# efficiency increase
A_vaild = np.array([[3,1],[1,2]])
b = np.array([9,8])

x =np.linalg.solve(A_vaild,b)
print("optimized solution: ",x)

Mathematic crash: Singular matrix
psedu-inverse :  [[0.04 0.08]
 [0.08 0.16]]
optimized solution:  [2. 3.]


In [5]:
np.random.seed(42)
# 1000 cars, 3 independent features
X_base = np.random.randn(1000, 3) 

# Feature 4 is 5x Feature 3 (Perfect collinearity!)
feature_4 = X_base[:, 2:3] * 5 
X = np.hstack((X_base, feature_4)) # Shape: (1000, 4)

# Target vector (Prices)
Y = np.random.randn(1000, 1)

X_T_X = X.T @ X
psuedu = np.linalg.pinv(X_T_X)

W = psuedu @ X.T @ Y
print(W)




[[ 0.0159614 ]
 [-0.02650382]
 [-0.00150009]
 [-0.00750043]]


### Topic 27: Eigenvalues & Eigenvectors Characteristic Equations

ML Math Notes: Eigen-Decomposition in Practice

**1. Covariance Matrices & PCA (Data Compression)**
*   **Concept:** Eigendecomposition of a Covariance matrix reveals how data spreads.
*   **Eigenvectors (Principal Components):** Point precisely in the directions of maximum variance.
*   **Eigenvalues:** Quantify the exact *amount* of variance (information) captured in that direction.
*   **ML Application:** Keep the top-$k$ eigenvectors to compress high-dimensional data (e.g., 1000D $\rightarrow$ 20D) while retaining >99% of the structural information.

**2. Hessian Matrices & Optimization (Loss Landscape)**
*   **Concept:** Eigendecomposition of a Hessian matrix (2nd-order derivatives) maps the geometric curvature of a Neural Network's loss function.
*   **All Eigenvalues $> 0$:** The optimizer has found a **Local Minimum** (a mathematical bowl).
*   **Mixed Eigenvalues ($+$ and $-$):** The optimizer is stuck in a **Saddle Point** (looks like a minimum from one axis, but a downward slope from another).
*   **ML Application:** Explains why basic Gradient Descent gets stuck, and why modern optimizers (like Adam) require momentum to escape saddle points.

In [ ]:
A = np.random.randn(4,4)
symmetric_matrix = A @ A.T
# eigh() returns eigenvalues in ASCENDING order (smallest to largest)
eigenvalues,eigenvector = np.linalg.eigh(symmetric_matrix)
print("eigenvalue : ",eigenvalues)
print("eigenvectors : ",eigenvector[: , -1])

# decending sorting for ml
sorted_indices = np.argsort(eigenvalues) [:: -1]
top_eigenvalues =eigenvalues[sorted_indices]
top_eigenvectors =eigenvector[:,sorted_indices]

eigenvalue :  [ 0.03673617  0.20515673  0.87662703 10.57001987]
eigenvectors :  [-0.56907283  0.25858089 -0.77152782 -0.11847723]


In [19]:
np.random.seed(42)
# 500 face images, 64 pixels each
X = np.random.randn(500, 64) 

# Step 1: Mean-center the data (already done for you)
X_centered = X - np.mean(X, axis=0)

# Step 2: Calculate the Covariance Matrix (64x64)
# Formula: (X^T @ X) / (N - 1)
N = X_centered.shape[0]
Cov_Matrix = (X_centered.T @ X_centered) / (N - 1)

eigenvalue,eigenvectors = np.linalg.eigh(Cov_Matrix)

sorted_indice = np.argsort(eigenvalue)[::-1]
decending_eigenvalues = eigenvalue[sorted_indice]
decending_eigenvectors = eigenvectors[:,sorted_indice]
slice_eigenvector = decending_eigenvectors[: ,0:3]
print(slice_eigenvector)
final = X_centered @ slice_eigenvector
print(final)



[[-0.04383056  0.20184409 -0.06343005]
 [ 0.02057897  0.05405324  0.05636917]
 [ 0.05809091  0.16969495  0.09602656]
 [ 0.26357346  0.02554394 -0.09993601]
 [-0.26177223 -0.04723487  0.01092082]
 [-0.06570264  0.15799942 -0.09150927]
 [ 0.09572427  0.16279894 -0.04754416]
 [ 0.21645397  0.00182405 -0.02709701]
 [-0.07379046 -0.02216911 -0.16802128]
 [-0.12263496 -0.0512678  -0.21921213]
 [ 0.0147645  -0.0222895  -0.01487139]
 [ 0.01359233 -0.01474865  0.25063098]
 [ 0.08249565 -0.02659437  0.17044101]
 [ 0.0868188  -0.07346894 -0.07134412]
 [-0.06900077 -0.03134351  0.0200616 ]
 [ 0.08061092 -0.16608012  0.08380647]
 [-0.17420251 -0.04860063  0.13050079]
 [ 0.06689765 -0.19309428  0.0878267 ]
 [-0.14438974 -0.07958056  0.19612151]
 [ 0.03854426 -0.06167029  0.12105011]
 [ 0.11936502  0.12021409 -0.09372248]
 [-0.16526599 -0.01245004  0.17005075]
 [ 0.00981124 -0.16544157 -0.08219681]
 [-0.15748658  0.07620186 -0.14735933]
 [ 0.02774654  0.08628047 -0.02945912]
 [ 0.28113613 -0.0079    